### 1. Prototype in its purest form

**The idea**
- *Instead of creating objects from a class, you create a new objects by copying an existing object*

So the **source of truth** is not the class - it's another object

### 2. Simplest possible Prototype

In [2]:
import copy

base = {
    "timeout": 30,
    "retries": 3
}

clone = copy.copy(base)

clone["timeout"] = 60

What happened:
- clone starts identical to base
- Then diverges
- base remains uncahnged

**This is already Prototype**

### 3. Shallow vs Deep Prototype

In [4]:
# Shallow copy

import copy

base = {
    "model": "gpt",
    "params": {
        "temperature": 0.2
    }
}

clone = copy.copy(base)
clone["params"]["temperature"] = 0.9

print(base["params"]["temperature"]) # Should be 0.2

0.9


In [5]:
# Deep copy

import copy

base = {
    "model": "gpt",
    "params": {
        "temperature": 0.2
    }
}

clone = copy.deepcopy(base)
clone["params"]["temperature"] = 0.9

print(base["params"]["temperature"]) # Correct

0.2


### Tule of thumb

- Immutable fiels -> shallow copy
- Mutable nested state -> deep copy required

### 4. Object-oriented Prototype

In [6]:
from abc import ABC, abstractmethod

class Prototype(ABC):
    @abstractmethod
    def clone(self) -> "Prototype": ...

This communicates intent:
- *Objects of this type can copy themselves*

In [7]:
import copy

class ModelConfig(Prototype):
    def __init__(self, model_id: str, params: dict):
        self.model_id = model_id
        self.params = params
        
    def clone(self):
        return copy.deepcopy(self)
    
base = ModelConfig("gpt-4.1-mini", {"temperature": 0.2})
experiment = base.clone()

experiment.params["temperature"] = 0.8

### Why this is good:
- Clone logic is encapsulated
- Callers don't think about copy mechanics

### 5. **Prototype for performane-heavy objects**

**Real problem**

You have an object that:
- loads files
- parses configs
- builds internal structures

**Creation is expensive**

5.1 Example: Heavy parse object

In [8]:
class HeavyParser:
    def __init__(self, rules: dict):
        print("Loading rules...")
        self.rules = rules
        self.cache = {}
        
    def parse(self, text: str):
        ...

In [9]:
# Instead of rebuilding
parser = HeavyParser({})

# Use prototype
base_parser = HeavyParser({})

def new_parser():
    return copy.copy(base_parser)

Loading rules...
Loading rules...


Why shallow copy may be enough:
- ruels are immutable
- cache can be reset or isolated

### 6. Prototype with partial reset

Ofter you want:
- copy structure
- reset runtime state

In [10]:
class Parser:
    def __init__(self, grammar, cache=None):
        self.grammar = grammar
        self.cache = cache or {}
        
    def clone(self):
        new = copy.copy(self)
        new.cache = {}
        return new

### 7.Prototype Registry 

You want:
- named tempaltes
- fast creation
- no conditional logic

In [11]:
class PrototypeRegistry:
    def __init__(self):
        self._registry = {}
        
    def register(self, name: str, prototype):
        self._registry[name] = prototype
        
    def create(self, name: str):
        if name not in self._registry:
            raise ValueError(f"Unknowm prootype {name}")
        return self._registry[name].clone()

In [13]:
registry = PrototypeRegistry()

registry.register(
    "fast",
    ModelConfig("gpt-4", {"temperature": 0.2})
)

registry.register(
    "creative",
    ModelConfig("gpt-4", {"temperature": 0.2})
)

cfg = registry.create("creative")